**MAESTRÍA EN INTELIGENCIA ARTIFICIAL APLICADA**

**Ciencia y analítica de datos · Grupo 10**

Tecnológico de Monterrey

**Profesora titular:** Dra. Grettel Barceló  
**Profesor tutor:** Mtro. Roberto Antonio Guevara González

**Actividad 3 · Acceso a bases de datos y manipulación de datos**

**Alumno:** Alejandro Santiago Baca Eyssautier  
**Matrícula:** A01656580

---

**Modelo relacional utilizado**

La tabla `tv_show` se relaciona de uno a muchos con `season`. A su vez, `view_summary` puede referirse a una película mediante `movie_id` o a una temporada mediante `season_id`. Esta estructura permite combinar la información descriptiva de cada título con sus métricas de visualización sin duplicar atributos generales.

In [1]:
%%capture
!pip -q install pymysql firebase-admin "sqlalchemy>=2.0"

In [2]:
import warnings
import pandas as pd
import sqlalchemy as sqla
import firebase_admin
from firebase_admin import credentials, firestore
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

**Conexión a MySQL**

La conexión utiliza el usuario de consulta proporcionado para el esquema `Netflix`. El parámetro `pool_recycle` evita mantener conexiones del pool durante más de una hora.

In [3]:
url = sqla.URL.create(
    "mysql+pymysql",
    username="mnaTC5053User",
    password="mnaTC5053Pass!",
    host="20.127.169.219",
    port=3306,
    database="Netflix"
)

db = sqla.create_engine(url, pool_recycle=3600)
conn = db.connect()

**Pregunta 1 · Películas con duración mayor a cinco horas**

Cinco horas equivalen a 300 minutos. La consulta conserva todas las columnas para no perder información descriptiva de las películas recuperadas.

In [4]:
query_1 = sqla.text("""
    SELECT *
    FROM movie
    WHERE runtime > 300
    ORDER BY runtime DESC, title
""")

peliculas_mas_cinco_horas = pd.read_sql(query_1, conn)
peliculas_mas_cinco_horas["available_globally"] = (
    peliculas_mas_cinco_horas["available_globally"].map(lambda valor: None if valor is None else int.from_bytes(valor, "big"))
) # pymysql doesnt convert bytes to int automatically
display(peliculas_mas_cinco_horas)

,id,created_date,modified_date,available_globally,locale,original_title,release_date,runtime,title
0,5793,2024-01-01,2024-01-01,0,None,日本統一シリーズ: 映画シリーズ,None,3892,Nihontouitsu Series: Film Series
1,5794,2024-01-01,2024-01-01,0,None,釣りバカ日誌: 映画シリーズ,None,2120,Free and Easy Series: Film Series
2,9729,2024-01-01,2024-01-01,0,None,織田同志会 織田征仁: 映画シリーズ,None,710,Seiji Oda: Film Series
3,9730,2024-01-01,2024-01-01,0,None,キングダム～首領になった男～: 映画シリーズ,None,427,Kingdom ~ The Man Who Became the Top ~: Film S...
4,5874,2024-01-01,2024-01-01,1,None,None,2021-08-06,312,Navarasa: Limited Series


**Análisis de resultado:** La consulta devuelve **5 películas** con duración mayor a 300 minutos: *Nihontouitsu Series: Film Series* (3,892 min, ≈ 65 horas), *Free and Easy Series: Film Series* (2,120 min), *Seiji Oda: Film Series* (710 min), *Kingdom ~ The Man Who Became the Top ~: Film Series* (427 min) y *Navarasa: Limited Series* (312 min). Por sus títulos, ninguno es un largometraje individual: son compilaciones o series limitadas registradas en la tabla `movie`, lo que explica duraciones tan extremas. Además, ninguno tiene `locale` registrado y cuatro de los cinco están disponibles únicamente en Estados Unidos (`available_globally = 0`).

**Pregunta 2 · Porcentaje de películas disponibles únicamente en Estados Unidos**

`available_globally = 0` identifica títulos que no están disponibles fuera de Estados Unidos. El denominador utiliza `COUNT(available_globally)`, por lo que los valores `NULL` quedan excluidos directamente en SQL.

In [5]:
query_2 = sqla.text("""
    SELECT ROUND(
        100.0 * SUM(
            CASE
                WHEN available_globally = 0 THEN 1
                ELSE 0
            END
        ) / COUNT(available_globally),
        2
    ) AS porcentaje_solo_eu
    FROM movie
    WHERE available_globally IS NOT NULL
""")

porcentaje_solo_eu = pd.read_sql(query_2, conn)
display(porcentaje_solo_eu)

,porcentaje_solo_eu
0,83.57


**Análisis de resultado:** El **83.57 %** de las películas con valor registrado en `available_globally` están disponibles únicamente en Estados Unidos. De las 11,787 películas de la tabla se excluyen 672 con valor `NULL`, por lo que el cálculo considera 11,115 registros: 9,289 solo en EU y 1,826 disponibles globalmente. Esto indica que la gran mayoría del catálogo registrado tiene distribución restringida a Estados Unidos.

**Pregunta 3 · Idiomas o regiones originales y valores faltantes**

La primera consulta muestra cada valor de `locale` junto con su frecuencia. La segunda calcula de forma independiente cuántos registros no tienen información de idioma o región.

In [6]:
query_3_locales = sqla.text("""
    SELECT locale, COUNT(*) AS cantidad
    FROM movie
    GROUP BY locale
    ORDER BY locale IS NULL, locale
""")

query_3_null = sqla.text("""
    SELECT COUNT(*) AS peliculas_locale_null
    FROM movie
    WHERE locale IS NULL
""")

locales_peliculas = pd.read_sql(query_3_locales, conn)
conteo_locale_null = pd.read_sql(query_3_null, conn)

display(locales_peliculas)
display(conteo_locale_null)

,locale,cantidad
0,en,532
1,None,11255


,peliculas_locale_null
0,11255


**Análisis de resultado:** El único idioma/región original registrado en `locale` es **`en`** (inglés), con 532 películas. Los demás **11,255 registros tienen `locale` con valor `NULL`**, es decir, el 95.5 % de las 11,787 películas. Por lo tanto, el campo `locale` está escasamente poblado y no permite conocer el idioma de la mayoría del catálogo.

**Pregunta 4 · Evaluación de la hipótesis sobre `locale` y los títulos**

La comparación no debe limitarse a contar valores faltantes. También es necesario distinguir entre títulos iguales, títulos diferentes y casos donde `original_title` está vacío. La tabla de contingencia permite identificar si las diferencias de título realmente coinciden con los `NULL` de `locale`.

In [7]:
query_4_conteos = sqla.text("""
    SELECT
        SUM(CASE WHEN locale IS NULL THEN 1 ELSE 0 END) AS locale_null,
        SUM(
            CASE
                WHEN original_title IS NOT NULL
                 AND BINARY title <> BINARY original_title
                THEN 1
                ELSE 0
            END
        ) AS titulos_diferentes
    FROM movie
""")

query_4_cruce = sqla.text("""
    SELECT
        CASE
            WHEN locale IS NULL THEN 'locale NULL'
            ELSE 'locale informado'
        END AS estado_locale,
        CASE
            WHEN original_title IS NULL THEN 'original_title NULL'
            WHEN BINARY title = BINARY original_title THEN 'titulos iguales'
            ELSE 'titulos diferentes'
        END AS relacion_titulos,
        COUNT(*) AS cantidad
    FROM movie
    GROUP BY estado_locale, relacion_titulos
    ORDER BY estado_locale, cantidad DESC
""")

query_4_restantes = sqla.text("""
    SELECT
        id,
        locale,
        title,
        original_title,
        CAST(available_globally AS UNSIGNED) AS available_globally,
        release_date,
        runtime
    FROM movie
    WHERE
        (
            locale IS NULL
            AND (
                original_title IS NULL
                OR BINARY title = BINARY original_title
            )
        )
        OR
        (
            locale IS NOT NULL
            AND original_title IS NOT NULL
            AND BINARY title <> BINARY original_title
        )
    ORDER BY locale IS NULL DESC, locale, title
""")

conteos_hipotesis = pd.read_sql(query_4_conteos, conn)
cruce_locale_titulos = pd.read_sql(query_4_cruce, conn)
registros_restantes = pd.read_sql(
    query_4_restantes,
    conn,
    dtype={"available_globally": "Int64"}
)

display(conteos_hipotesis)
display(cruce_locale_titulos)
display(registros_restantes)

,locale_null,titulos_diferentes
0,"11,255.00","3,964.00"


,estado_locale,relacion_titulos,cantidad
0,locale informado,original_title NULL,532
1,locale NULL,original_title NULL,7291
2,locale NULL,titulos diferentes,3964


,id,locale,title,original_title,available_globally,release_date,runtime
0,7056,None,"""Sr.""",None,1,2022-12-02,89
1,11542,None,#LadyRancho,None,<NA>,None,91
2,2308,None,#realityhigh,None,1,2017-09-08,99
3,7057,None,#Selfie 69,None,1,None,119
4,3485,None,'71,None,0,None,100
...,...,...,...,...,...,...,...
7286,1657,None,비상선언,None,0,None,122
7287,11115,None,선생 김봉두,None,0,None,117
7288,9358,None,아이 캔 스피크,None,0,None,119
7289,5938,None,침묵,None,0,None,125


**Análisis de resultado:** Los conteos **no coinciden**: hay 11,255 películas con `locale` nulo, pero solo **3,964 películas tienen `title` y `original_title` diferentes**. La comparación se hace con `BINARY` para distinguir acentos y caracteres especiales; por ejemplo, *Veronica* frente a *Verónica* o *Malefique* frente a *Maléfique* son 17 casos que una comparación insensible a acentos consideraría iguales (3,947 en lugar de 3,964).

La tabla de contingencia muestra que:

- Las 3,964 películas con títulos diferentes tienen `locale` nulo y corresponden efectivamente a títulos en otros idiomas (p. ej., *Society of the Snow* / *La sociedad de la nieve*, *Under Paris* / *Sous la Seine*).
- Los **7,291 registros restantes** con `locale` nulo tienen `original_title` también en `NULL`, así que no es posible comparar los títulos. Estas son las características de los registros que no coinciden y corresponden a la tabla de registros restantes.
- Las 532 películas con `locale = 'en'` tampoco tienen `original_title`.
- No existe ninguna película con `locale` nulo cuyo título original coincida exactamente con el título principal.

**Conclusión:** la suposición **no es válida** como regla general. Solo el 35.2 % de los registros con `locale` nulo (3,964 de 11,255) muestra evidencia de un idioma distinto del inglés; el 64.8 % restante carece de título original y, por lo tanto, el valor nulo refleja ausencia de metadatos y no un idioma específico. Un `NULL` en `locale` no debe interpretarse como "otro idioma".

**Pregunta 5 · Película con mayor permanencia en el top 10**

Se utiliza el máximo de `cumulative_weeks_in_top10` considerando todos los registros de `view_summary` asociados a películas, porque esa variable representa directamente la permanencia acumulada en la lista.

In [8]:
query_5 = sqla.text("""
    SELECT
        m.title,
        MAX(v.cumulative_weeks_in_top10) AS semanas_en_top_10
    FROM movie AS m
    INNER JOIN view_summary AS v
        ON v.movie_id = m.id
    GROUP BY m.id, m.title
    ORDER BY semanas_en_top_10 DESC, m.title
    LIMIT 1
""")

pelicula_mayor_permanencia = pd.read_sql(query_5, conn)
display(pelicula_mayor_permanencia)

,title,semanas_en_top_10
0,The Boss Baby,22


**Análisis de resultado:** La película que ha permanecido más tiempo en el top 10 es ***The Boss Baby***, con **22 semanas acumuladas**. Solo los registros semanales (`WEEKLY`) tienen valor en `cumulative_weeks_in_top10`; en los semestrales el campo es `NULL`, por lo que no influyen en el máximo.

**Pregunta 6 · Programas de televisión con mayor cantidad de temporadas**

La relación uno a muchos entre `tv_show` y `season` permite contar las temporadas asociadas a cada programa y ordenar los resultados de mayor a menor.

In [9]:
query_6 = sqla.text("""
    SELECT
        t.title AS programa,
        COUNT(s.id) AS cantidad_temporadas
    FROM tv_show AS t
    INNER JOIN season AS s
        ON s.tv_show_id = t.id
    GROUP BY t.id, t.title
    ORDER BY cantidad_temporadas DESC, programa
    LIMIT 5
""")

programas_mas_temporadas = pd.read_sql(query_6, conn)
display(programas_mas_temporadas)

,programa,cantidad_temporadas
0,Grey's Anatomy,20
1,Naruto Shippuden,20
2,Heartland (2007),17
3,It's Always Sunny in Philadelphia,16
4,NCIS,15


**Análisis de resultado:** Los 5 programas con más temporadas son ***Grey's Anatomy*** y ***Naruto Shippuden*** (20 temporadas cada uno), ***Heartland (2007)*** (17), ***It's Always Sunny in Philadelphia*** (16) y ***NCIS*** (15). En el quinto lugar existe un **empate a 15 temporadas** entre *NCIS*, *Raw* y *Supernatural (2005)*; el criterio de desempate alfabético de la consulta deja a *NCIS* en la lista, pero los tres programas tienen la misma cantidad de temporadas.

**Pregunta 7 · Intervalos de los resúmenes semestrales**

Los pares distintos de `start_date` y `end_date` muestran los periodos cubiertos por los reportes de duración semestral.

In [10]:
query_7 = sqla.text("""
    SELECT DISTINCT
        start_date,
        end_date
    FROM view_summary
    WHERE duration = 'SEMI_ANNUALLY'
    ORDER BY start_date
""")

intervalos_semestrales = pd.read_sql(query_7, conn)
display(intervalos_semestrales)

,start_date,end_date
0,2023-07-01,2023-12-31
1,2024-01-01,2024-06-30


**Análisis de resultado:** Existen **dos intervalos semestrales** en `view_summary`: del **2023-07-01 al 2023-12-31** (segundo semestre de 2023) y del **2024-01-01 al 2024-06-30** (primer semestre de 2024). El segundo intervalo es el que corresponde al "primer período semestral de 2024" utilizado en la pregunta 8.

**Pregunta 8 · Vistas de las temporadas de Grey's Anatomy en el primer semestre de 2024**

La consulta enlaza el programa, sus temporadas y sus métricas semestrales. Las temporadas se ordenan por `views` de mayor a menor.

In [11]:
query_8 = sqla.text("""
    SELECT
        s.season_number,
        s.title AS season_title,
        s.runtime,
        v.views,
        v.hours_viewed,
        v.view_rank,
        v.start_date,
        v.end_date
    FROM tv_show AS t
    INNER JOIN season AS s
        ON s.tv_show_id = t.id
    INNER JOIN view_summary AS v
        ON v.season_id = s.id
    WHERE t.title = :programa
      AND v.duration = 'SEMI_ANNUALLY'
      AND v.start_date = '2024-01-01'
    ORDER BY v.views DESC, s.season_number
""")

greys_sql = pd.read_sql(
    query_8,
    conn,
    params={"programa": "Grey's Anatomy"}
)

display(greys_sql)

,season_number,season_title,runtime,views,hours_viewed,view_rank,start_date,end_date
0,1,Grey's Anatomy: Season 1,387,3600000,23400000,None,2024-01-01,2024-06-30
1,2,Grey's Anatomy: Season 2,1163,3100000,60600000,None,2024-01-01,2024-06-30
2,3,Grey's Anatomy: Season 3,1097,2900000,53700000,None,2024-01-01,2024-06-30
3,4,Grey's Anatomy: Season 4,742,2900000,35300000,None,2024-01-01,2024-06-30
4,5,Grey's Anatomy: Season 5,1035,2900000,49300000,None,2024-01-01,2024-06-30
5,6,Grey's Anatomy: Season 6,1032,2800000,48000000,None,2024-01-01,2024-06-30
6,7,Grey's Anatomy: Season 7,949,2700000,43000000,None,2024-01-01,2024-06-30
7,8,Grey's Anatomy: Season 8,1033,2600000,44700000,None,2024-01-01,2024-06-30
8,9,Grey's Anatomy: Season 9,1032,2500000,43300000,None,2024-01-01,2024-06-30
9,10,Grey's Anatomy: Season 10,1035,2400000,41700000,None,2024-01-01,2024-06-30


**Análisis de resultado:** Se obtienen las **20 temporadas** de *Grey's Anatomy* con vistas en el primer semestre de 2024. La **temporada 1** encabeza la lista con **3.6 millones** de vistas, seguida de la temporada 2 (3.1 M) y de las temporadas 3, 5 y 4 (2.9 M cada una). A partir de ahí las vistas descienden gradualmente hasta estabilizarse alrededor de 1.7–2.0 M entre las temporadas 12 y 19. La **temporada 20** queda al final con solo **100 mil** vistas.

**Interpretación:** el patrón refleja el comportamiento típico de una serie de catálogo con narrativa secuencial. Los nuevos espectadores, y quienes la vuelven a ver, comienzan por la primera temporada, y una parte de la audiencia abandona conforme avanza, lo que produce una caída de aproximadamente 44 % entre la temporada 1 y la 12. La meseta de cerca de 2 millones en las temporadas finales sugiere un núcleo de audiencia fiel que sigue la serie completa; la temporada 19 (2.0 M) supera ligeramente a las temporadas 15–18, lo que podría indicar un efecto de novedad. La temporada 20 no es comparable con el resto: su duración disponible (431 min frente a ~1,000 min de las demás) indica que solo estaba disponible parcialmente durante el período. También se observa que `views` no equivale a horas vistas: la temporada 2 acumula más horas (60.6 M) que la 1 (23.4 M) por su mayor duración. Por ello, `views` mide mejor la cantidad de audiencia que llegó a cada temporada.

**Pregunta 9 · Replicación de la consulta con lógica relacional de Pandas**

Los `merge` reproducen los `INNER JOIN` de SQL. Después se aplican los filtros del programa, la duración y la fecha inicial, y finalmente se ordenan las vistas.

In [12]:
tv_show = pd.read_sql(sqla.text("SELECT * FROM tv_show"), conn)
season = pd.read_sql(sqla.text("SELECT * FROM season"), conn)
view_summary = pd.read_sql(sqla.text("SELECT * FROM view_summary"), conn)

tv_show_rel = tv_show.rename(
    columns={"id": "tv_show_id", "title": "tv_show_title"}
)
season_rel = season.rename(
    columns={"id": "season_id", "title": "season_title"}
)
view_rel = view_summary.copy()
view_rel["start_date"] = pd.to_datetime(view_rel["start_date"])
view_rel["end_date"] = pd.to_datetime(view_rel["end_date"])

resultado_pandas = (
    season_rel
    .merge(
        tv_show_rel[["tv_show_id", "tv_show_title"]],
        on="tv_show_id",
        how="inner"
    )
    .merge(
        view_rel[
            [
                "season_id",
                "duration",
                "start_date",
                "end_date",
                "views",
                "hours_viewed",
                "view_rank"
            ]
        ],
        on="season_id",
        how="inner"
    )
    .loc[
        lambda datos:
            datos["tv_show_title"].eq("Grey's Anatomy")
            & datos["duration"].eq("SEMI_ANNUALLY")
            & datos["start_date"].eq(pd.Timestamp("2024-01-01")),
        [
            "season_number",
            "season_title",
            "runtime",
            "views",
            "hours_viewed",
            "view_rank",
            "start_date",
            "end_date"
        ]
    ]
    .sort_values(
        ["views", "season_number"],
        ascending=[False, True]
    )
    .astype({"season_number": "int64"})
    .reset_index(drop=True)
)

display(resultado_pandas)

,season_number,season_title,runtime,views,hours_viewed,view_rank,start_date,end_date
0,1,Grey's Anatomy: Season 1,387,3600000,23400000,NaN,2024-01-01,2024-06-30
1,2,Grey's Anatomy: Season 2,1163,3100000,60600000,NaN,2024-01-01,2024-06-30
2,3,Grey's Anatomy: Season 3,1097,2900000,53700000,NaN,2024-01-01,2024-06-30
3,4,Grey's Anatomy: Season 4,742,2900000,35300000,NaN,2024-01-01,2024-06-30
4,5,Grey's Anatomy: Season 5,1035,2900000,49300000,NaN,2024-01-01,2024-06-30
5,6,Grey's Anatomy: Season 6,1032,2800000,48000000,NaN,2024-01-01,2024-06-30
6,7,Grey's Anatomy: Season 7,949,2700000,43000000,NaN,2024-01-01,2024-06-30
7,8,Grey's Anatomy: Season 8,1033,2600000,44700000,NaN,2024-01-01,2024-06-30
8,9,Grey's Anatomy: Season 9,1032,2500000,43300000,NaN,2024-01-01,2024-06-30
9,10,Grey's Anatomy: Season 10,1035,2400000,41700000,NaN,2024-01-01,2024-06-30


**Análisis de resultado:** El resultado con Pandas reproduce exactamente la tabla de la pregunta 8: las mismas 20 temporadas, en el mismo orden y con los mismos valores. Los dos `merge` con `how="inner"` equivalen a los `INNER JOIN` entre `season`, `tv_show` y `view_summary`. El filtro con `.loc` sustituye al `WHERE` y `sort_values` sustituye al `ORDER BY`. `season_number` se convierte a entero porque, en la tabla completa `season`, la columna contiene valores nulos y Pandas la carga como `float`.

In [13]:
columnas_comparacion = [
    "season_number",
    "season_title",
    "runtime",
    "views",
    "hours_viewed",
    "view_rank",
    "start_date",
    "end_date"
]

sql_comparable = greys_sql[columnas_comparacion].copy()
sql_comparable["start_date"] = pd.to_datetime(sql_comparable["start_date"])
sql_comparable["end_date"] = pd.to_datetime(sql_comparable["end_date"])
sql_comparable = (
    sql_comparable
    .astype(resultado_pandas[columnas_comparacion].dtypes.to_dict())
    .reset_index(drop=True)
)

comparacion = pd.DataFrame(
    {
        "coinciden_sql_y_pandas": [
            sql_comparable.equals(resultado_pandas[columnas_comparacion])
        ]
    }
)

display(comparacion)

,coinciden_sql_y_pandas
0,True


**Análisis de resultado:** La comparación devuelve **`True`**: los dataframes obtenidos con SQL y con Pandas son idénticos en valores, orden y tipos de datos. Esto confirma que la lógica relacional de Pandas replica correctamente la consulta 8.

**Pregunta 10 · Acceso a Firestore y lectura de la colección EMPLOYEE**

En Firestore, a diferencia de MySQL donde se utiliza un usuario y contraseña para conectarse, la autenticación se realiza mediante un archivo JSON que almacena las credenciales necesarias para acceder a la base de datos. El archivo `consultancy.json` almacena la clave privada para autenticar una cuenta y autorizar el acceso a los servicios. A través de la función Certificate(), se regresa una credencial inicializada, que puedes utilizar para crear una nueva instancia de la aplicación

In [14]:
if firebase_admin._apps:
    firebase_admin.delete_app(firebase_admin.get_app())

cred = credentials.Certificate("consultancy.json")
firebase_admin.initialize_app(cred)
db_firestore = firestore.client()

registros = []
for documento in db_firestore.collection("EMPLOYEE").stream():
    registro = documento.to_dict()
    registro["id"] = documento.id
    registros.append(registro)

employee_df = pd.DataFrame(registros)
columnas_employee = ["id"] + [
    columna
    for columna in employee_df.columns
    if columna != "id"
]
employee_df = employee_df[columnas_employee]

display(employee_df)

,id,emp_fname,emp_hiredate,emp_lname
0,8LcLuxVHGAd2d9IQc5jF,David,1989-07-12 06:00:00+00:00,Senior
1,Fzd60D6Z2CU4n0wVV8YN,William,2004-06-04 05:00:00+00:00,Smithfield
2,lX5xuQ5w3i6ib2ExccWY,John,2000-11-08 06:00:00+00:00,News
3,yocFj2lichOkbAj9NBfp,June,1996-12-01 06:00:00+00:00,Arbough


**Análisis de resultado:** La autenticación con `consultancy.json` fue exitosa y la colección `EMPLOYEE` contiene **4 documentos**, mostrados en el dataframe con la columna `id` (identificador del documento en Firestore) en primer lugar, seguida de `emp_hiredate`, `emp_lname` y `emp_fname`. Los empleados son David Senior (contratado el 1989-07-12), William Smithfield (2004-06-04), John News (2000-11-08) y June Arbough (1996-12-01). A diferencia de MySQL, cada documento se obtiene como diccionario mediante `to_dict()`, y el `id` debe agregarse explícitamente porque no forma parte de los campos almacenados.

**Cierre de conexiones**

In [15]:
if firebase_admin._apps:
    firebase_admin.delete_app(firebase_admin.get_app())

conn.close()
db.dispose()